### XAI- Interpret ML Tutorial
```bash
pip install interpret
```

#### Types of Models Based on Interpretation or Explainability
+ Black Box Model Explanations
    - input --> [BlackBox Model] --> Output
    - we don't know how the model comes to it decisions
    - we use external packages to explain the prediction of the model
  
+ GlassBox Model
    - input --> [GlassBox Model] --> Output
    - Models are inherently interpretable
    - We know how the model makes decision
    - Like a Glass it is transparent and visible
    


In [ ]:
! pip install interpret

In [ ]:
# Load EDA Pkgs
import pandas as pd
import numpy as np

In [ ]:
# Load Dataset
df = pd.read_csv("/content/bank-full.csv",sep=';')

In [ ]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [ ]:
# Check for Datatype
df.dtypes

age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

In [ ]:
df1 = pd.DataFrame({col: df[col].astype('category').cat.codes for col in df}, index=df.index)


#### Encoding

In [ ]:
df1.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,40,4,1,2,0,3036,1,0,2,4,8,261,0,0,0,3,0
1,26,9,2,1,0,945,1,0,2,4,8,151,0,0,0,3,0
2,15,2,1,1,0,918,1,1,2,4,8,76,0,0,0,3,0
3,29,1,1,3,0,2420,1,0,2,4,8,92,0,0,0,3,0
4,15,11,2,3,0,917,0,0,2,4,8,198,0,0,0,3,0


In [ ]:
# Features and Ylabels
Xfeatures = df1[['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome']]
ylabels = df1['y']

In [ ]:
# Split Dataset
x_train,x_test,y_train,y_test = train_test_split(Xfeatures,ylabels,test_size=0.3,random_state=7)

### Build Model

In [ ]:
# ML Pkgs
from sklearn.linear_model import LogisticRegression
# Metrics
from sklearn.model_selection import train_test_split,cross_val_score

In [ ]:
# Log Reg Model
lr_model = LogisticRegression()
lr_model.fit(x_train,y_train)

/usr/local/lib/python3.7/dist-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


LogisticRegression()

In [ ]:
# Accuracy of Model
lr_model.score(x_test,y_test)

0.8905927455028015

#### Build A GlassBox Model
+ EBM (Explainable Boosting Classifier)

In [ ]:
import interpret

In [ ]:
# Methods/Attrib
dir(interpret)

['NullHandler',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 'getLogger',
 'get_show_addr',
 'get_visualize_provider',
 'init_show_server',
 'preserve',
 'provider',
 'set_show_addr',
 'set_visualize_provider',
 'show',
 'show_link',
 'shutdown_show_server',
 'status_show_server',
 'utils',
 'version',
 'visual']

In [ ]:
# Our EBM Model Glassbox model
from interpret.glassbox import ExplainableBoostingClassifier

In [ ]:
ebm = ExplainableBoostingClassifier()
ebm.fit(x_train,y_train)

ExplainableBoostingClassifier(feature_names=['age', 'job', 'marital',
                                             'education', 'default', 'balance',
                                             'housing', 'loan', 'contact',
                                             'day', 'month', 'duration',
                                             'campaign', 'pdays', 'previous',
                                             'poutcome', 'housing x month',
                                             'housing x duration',
                                             'day x month', 'month x duration',
                                             'contact x duration',
                                             'duration x poutcome',
                                             'age x duration',
                                             'balance x duration',
                                             'duration x pdays',
                                             'dur...
                  

In [ ]:
# Accuracy of EBM
ebm.score(x_test,y_test)

0.9081391919787674

In [ ]:
### Single Prediction
ex1 = x_test.iloc[8]
act1 = y_test.iloc[8]

In [ ]:
# Prediction with EBM
print(ebm.predict([ex1]))
print(ebm.predict_proba([ex1]))

[0]
[[0.93202639 0.06797361]]


#### Model Interpretation

In [ ]:
from interpret import show

In [ ]:
# Global Explanation

In [ ]:
ebm_global = ebm.explain_global()

In [ ]:
# method 1:
show(ebm_global)

/usr/local/lib/python3.7/dist-packages/interpret/provider/visualize.py:44: UserWarning: Cloud environment detected (['colab', 'ipython']): viz integration is still experimental.
  detected_envs


In [ ]:
# Local Explanation

In [ ]:
ebm_local = ebm.explain_local(x_test,y_test)

In [ ]:
show(ebm_local)

In [ ]:
ebm_local = ebm.explain_local(x_test, y_test)
show(ebm_local)